# Save Your Work

Before starting, save this notebook to your Google Drive:
1. Click **File** → **Save a copy in Drive**
2. The copy will open automatically
3. Work in the Google Drive copy from now on

---

# Binary Classification

Build and evaluate classifiers that predict one of two outcomes.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve
)

## Step 1: Load and Prepare Data

In [ ]:
# Load Titanic dataset
df = sns.load_dataset('titanic')
df = df[['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare']].dropna()

# Encode sex
df['sex'] = df['sex'].map({'male': 0, 'female': 1})

print(f"Data shape: {df.shape}")
print(f"\nTarget distribution:")
print(df['survived'].value_counts())
print(f"\nFirst rows:")
print(df.head())

## Step 2: Train/Test Split

In [ ]:
X = df.drop(columns='survived')
y = df['survived']

# Split with stratification to maintain class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training data scaled. Shape: {X_train_scaled.shape}")

# Logistic regression: classic binary classifier
clf = LogisticRegression(random_state=42, max_iter=1000)
clf.fit(X_train_scaled, y_train)

print("Model trained!")
print(f"\nModel coefficients:")
for feature, coef in zip(X.columns, clf.coef_[0]):
    print(f"  {feature}: {coef:.4f}")

# Hard predictions (0 or 1)
y_pred = clf.predict(X_test_scaled)

# Probability predictions
y_pred_proba = clf.predict_proba(X_test_scaled)[:, 1]  # probability of class 1

print(f"Hard predictions (first 10): {y_pred[:10]}")
print(f"\nProbabilities (first 10): {y_pred_proba[:10]}")

## Step 6: Evaluate with Key Metrics

In [ ]:
# Compute metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print("=== Classification Metrics ===")
print(f"Accuracy:  {accuracy:.4f}  (overall correctness)")
print(f"Precision: {precision:.4f}  (of predicted positive, how many were correct?)")
print(f"Recall:    {recall:.4f}    (of actual positive, how many did we catch?)")
print(f"F1-Score:  {f1:.4f}      (balance between precision and recall)")
print(f"ROC-AUC:   {roc_auc:.4f}   (across all thresholds)")

## Step 7: Confusion Matrix

In [ ]:
# Confusion Matrix: shows all 4 outcomes
# TP: true positive, FP: false positive
# TN: true negative, FN: false negative

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)
print(f"\nTrue Negatives:  {cm[0, 0]}")
print(f"False Positives: {cm[0, 1]}")
print(f"False Negatives: {cm[1, 0]}")
print(f"True Positives:  {cm[1, 1]}")

## Step 8: ROC Curve

In [ ]:
# ROC curve shows performance across all thresholds
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve for Binary Classifier')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()